In [ ]:
!pip install -q yfinance ta streamlit tensorflow scikit-learn matplotlib pyngrok


  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 77.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 83.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 6.3 MB/s eta 0:00:00


In [ ]:
!ngrok config add-authtoken 2vPEsbKGn5z4tMJ5dyeDs0PqiLt_4aGe8YkuKCCcbiZAcz3Yj

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [ ]:
%%writefile app.py
import streamlit as st
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, regularizers
from sklearn.preprocessing import MinMaxScaler
import ta

st.title("📈 Stock Prediction with TCN + Transformer")

# Sidebar
st.sidebar.header("Configuration")
ticker = st.sidebar.text_input("Stock Ticker", value="TSLA")
start_date = st.sidebar.date_input("Start Date", value=pd.to_datetime("2010-01-01"))
end_date = st.sidebar.date_input("End Date", value=pd.to_datetime("2024-01-01"))

@st.cache_data
def load_data(ticker, start, end):
    data = yf.download(ticker, start=start, end=end)

    # Extract Close as a proper Series, guaranteed to be 1D
    close = data['Close']

    # Make sure it's NOT a 2D array (this is the real fix)
    if isinstance(close, pd.DataFrame):
        close = close.squeeze()
    elif isinstance(close, np.ndarray):
        close = pd.Series(close.ravel(), index=data.index)

    # Technical Indicators
    data['SMA_20'] = close.rolling(window=20).mean()
    data['EMA_20'] = close.ewm(span=20).mean()

    # ✅ FIX: Ensure we pass a 1D Series to RSIIndicator
    data['RSI'] = ta.momentum.RSIIndicator(close=close.astype(float), window=14).rsi()

    # Bollinger Bands
    rolling_std = close.rolling(window=20).std()
    data['Bollinger_Upper'] = data['SMA_20'] + 2 * rolling_std
    data['Bollinger_Lower'] = data['SMA_20'] - 2 * rolling_std

    # Handle NaNs
    data.ffill(inplace=True)
    data.dropna(inplace=True)

    return data





data = load_data(ticker, start_date, end_date)

st.subheader("📊 Stock Prices and Indicators")
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(data['Close'], label='Close')
ax.plot(data['SMA_20'], label='SMA 20', linestyle='--')
ax.plot(data['Bollinger_Upper'], label='Upper BB', linestyle='--')
ax.plot(data['Bollinger_Lower'], label='Lower BB', linestyle='--')
ax.set_title("Price with SMA & Bollinger Bands")
ax.legend()
st.pyplot(fig)

features = ['Open', 'High', 'Low', 'Close', 'Volume',
            'SMA_20', 'EMA_20', 'RSI', 'Bollinger_Upper', 'Bollinger_Lower']
scaler = MinMaxScaler()
scaled_data = pd.DataFrame(scaler.fit_transform(data[features]), columns=features, index=data.index)

def create_sequences(data, sequence_length=60, target='Close'):
    X, y = [], []
    for i in range(sequence_length, len(data)):
        X.append(data.iloc[i-sequence_length:i].drop(columns=[target]).values)
        y.append(data.iloc[i][target])
    return np.array(X), np.array(y)

X, y = create_sequences(scaled_data, sequence_length=60, target='Close')
split_1 = int(0.7 * len(X))
split_2 = int(0.85 * len(X))
X_train, y_train = X[:split_1], y[:split_1]
X_val, y_val = X[split_1:split_2], y[split_1:split_2]
X_test, y_test = X[split_2:], y[split_2:]

def TCN_Block(input_shape):
    inputs = tf.keras.Input(shape=input_shape)
    x = inputs
    for rate in [1, 2, 4, 8]:
        x = layers.Conv1D(64, 3, dilation_rate=rate, padding='causal', activation='relu',
                          kernel_regularizer=regularizers.l2(1e-4))(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(0.2)(x)
    return tf.keras.Model(inputs, x, name="TCN_Block")

def Transformer_Block(input_shape):
    inputs = tf.keras.Input(shape=input_shape)
    attn_output = layers.MultiHeadAttention(num_heads=4, key_dim=input_shape[-1])(inputs, inputs)
    attn_output = layers.Dropout(0.2)(attn_output)
    out1 = layers.LayerNormalization()(inputs + attn_output)
    ff = layers.Dense(64, activation='relu')(out1)
    ff = layers.Dense(input_shape[-1])(ff)
    out2 = layers.LayerNormalization()(out1 + ff)
    return tf.keras.Model(inputs, out2, name="Transformer_Block")

def build_model(input_shape):
    inputs = tf.keras.Input(shape=input_shape)
    x = TCN_Block(input_shape)(inputs)
    x = Transformer_Block((x.shape[1], x.shape[2]))(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(64, activation='relu', kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(1)(x)
    return tf.keras.Model(inputs, outputs)

st.subheader("🔧 Model Training")
if st.button("Train Model"):
    model = build_model(X_train.shape[1:])
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    model.fit(X_train, y_train,
              validation_data=(X_val, y_val),
              epochs=20,
              batch_size=32,
              verbose=0,
              callbacks=[
                  tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
                  tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3)
              ])
    st.success("✅ Model trained!")

    # Predictions
    y_pred = model.predict(X_test)

    # Enhanced plot
    import matplotlib.pyplot as plt
    import numpy as np

    st.subheader("📈 Enhanced Prediction Chart")

    index = np.arange(len(y_test))

    fig, ax = plt.subplots(figsize=(14, 8))
    ax.plot(index, y_test, label="Actual", color='blue', linewidth=1.5)
    ax.plot(index, y_pred, label="Predicted", color='orange', linestyle='--', linewidth=1.5)

    # Gridlines
    ax.grid(color='gray', linestyle='--', linewidth=0.5, alpha=0.7)

    # Fill between error
    ax.fill_between(index, y_test.flatten(), y_pred.flatten(), color='gray', alpha=0.2, label="Error")

    # Labels and legend
    ax.set_title("Actual vs Predicted Stock Prices", fontsize=16, fontweight='bold')
    ax.set_xlabel("Sample Index", fontsize=12)
    ax.set_ylabel("Stock Price (Normalized)", fontsize=12)
    ax.legend(loc="upper right", fontsize=12)

    # Optional zoom
    start, end = 50, 100
    ax.set_xlim(start, end)

    # Tight layout and display
    plt.tight_layout()
    st.pyplot(fig)


    from sklearn.metrics import r2_score
    r2 = r2_score(y_test, y_pred)
    st.metric("R² Score", f"{r2:.4f}")


Writing app.py


In [ ]:
from pyngrok import ngrok
import os

# Kill any existing tunnels (just in case)
ngrok.kill()

# Start Streamlit via ngrok
public_url = ngrok.connect(8501)
print(f"🌐 Open your app here: {public_url}")

# Run the Streamlit app
!streamlit run app.py &



🌐 Open your app here: NgrokTunnel: "https://ffd1-34-16-140-84.ngrok-free.app" -> "http://localhost:8501"



  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.16.140.84:8501

2025-04-09 05:16:09.781209: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1744175769.804867    1688 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1744175769.813074    1688 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-04-09 05:16:09.837836: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in per

In [ ]:
%%writefile app.py
import streamlit as st
import yfinance as yf
import pandas as pd
import numpy as np
import ta
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
import tensorflow as tf
from tensorflow.keras import layers, regularizers
import matplotlib.pyplot as plt

# ========================
# Utility Functions
# ========================
def download_stock_data(ticker, start, end):
    df = yf.download(ticker, start=start, end=end)
    df.columns = pd.MultiIndex.from_product([[\"\"], df.columns])  # mimic multi-index for compatibility
    return df

def preprocess_data(data):
    close_prices = data[('Close', '')]
    data[('SMA_20', '')] = close_prices.rolling(window=20).mean()
    data[('EMA_20', '')] = close_prices.ewm(span=20, adjust=False).mean()
    data[('RSI', '')] = ta.momentum.RSIIndicator(close=close_prices, window=14).rsi()
    std = close_prices.rolling(window=20).std()
    data[('Bollinger_Upper', '')] = data[('SMA_20', '')] + 2 * std
    data[('Bollinger_Lower', '')] = data[('SMA_20', '')] - 2 * std
    data.columns = [col[0] if isinstance(col, tuple) else col for col in data.columns]
    data.ffill(inplace=True)
    data.dropna(inplace=True)
    return data

def normalize_features(data):
    features = ['Open', 'High', 'Low', 'Close', 'Volume',
                'SMA_20', 'EMA_20', 'RSI', 'Bollinger_Upper', 'Bollinger_Lower']
    scaler = MinMaxScaler()
    data[features] = scaler.fit_transform(data[features])
    return data

def create_sequences(data, sequence_length, target_column='Close'):
    X, y = [], []
    for i in range(sequence_length, len(data)):
        features = data.iloc[i-sequence_length:i].drop(columns=[target_column]).values
        X.append(features)
        y.append(data.iloc[i][target_column])
    return np.array(X), np.array(y)

def TCN_Block(input_shape, num_filters=64, kernel_size=3, dilation_rates=[1, 2, 4, 8], dropout_rate=0.2):
    inputs = tf.keras.Input(shape=input_shape)
    x = inputs
    for rate in dilation_rates:
        x = layers.Conv1D(filters=num_filters, kernel_size=kernel_size, dilation_rate=rate,
                          padding='causal', activation='relu', kernel_regularizer=regularizers.l2(1e-4))(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(dropout_rate)(x)
    return tf.keras.Model(inputs, x)

def Transformer_Block(input_shape, num_heads=4, ff_dim=64, dropout_rate=0.2):
    inputs = tf.keras.Input(shape=input_shape)
    attention = layers.MultiHeadAttention(num_heads=num_heads, key_dim=input_shape[-1])(inputs, inputs)
    attention = layers.Dropout(dropout_rate)(attention)
    attention = layers.LayerNormalization()(inputs + attention)
    ff = layers.Dense(ff_dim, activation='relu')(attention)
    ff = layers.Dropout(dropout_rate)(ff)
    ff = layers.Dense(input_shape[-1])(ff)
    ff = layers.LayerNormalization()(attention + ff)
    return tf.keras.Model(inputs, ff)

def build_model(input_shape):
    inputs = tf.keras.Input(shape=input_shape)
    x = TCN_Block(input_shape)(inputs)
    x = Transformer_Block((x.shape[1], x.shape[2]))(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(64, activation='relu', kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(1)(x)
    return tf.keras.Model(inputs, outputs)

# ========================
# Streamlit UI
# ========================
st.title(\"📈 TSLA Stock Prediction using TCN + Transformer\")

ticker = st.text_input(\"Enter Ticker Symbol\", \"TSLA\")

if st.button(\"Train Model\"):

    data = download_stock_data(ticker, '2010-01-01', '2024-01-01')
    data = preprocess_data(data)
    data = normalize_features(data)
    X, y = create_sequences(data, sequence_length=60)

    X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.15, random_state=42)
    X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.18, random_state=42)

    model = build_model(X_train.shape[1:])
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])

    model.fit(X_train, y_train,
              validation_data=(X_val, y_val),
              epochs=50,
              batch_size=32,
              callbacks=[
                  tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
                  tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5)
              ],
              verbose=0)

    y_pred = model.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    st.success(f\"Model trained successfully! R² Score: {r2:.4f}\")

    # Plot
    index = np.arange(len(y_test))
    fig, ax = plt.subplots(figsize=(14, 8))
    ax.plot(index, y_test, label=\"Actual\", color='blue', linewidth=1.5)
    ax.plot(index, y_pred, label=\"Predicted\", color='orange', linestyle='--', linewidth=1.5)
    ax.grid(color='gray', linestyle='--', linewidth=0.5, alpha=0.7)
    ax.fill_between(index, y_test.flatten(), y_pred.flatten(), color='gray', alpha=0.2, label=\"Error\")
    ax.set_title(\"Actual vs Predicted Stock Prices\", fontsize=16, fontweight='bold')
    ax.set_xlabel(\"Sample Index\", fontsize=12)
    ax.set_ylabel(\"Stock Price (Normalized)\", fontsize=12)
    ax.legend(loc=\"upper right\", fontsize=12)
    ax.set_xlim(50, 100)
    plt.tight_layout()
    st.pyplot(fig)
